<a href="https://colab.research.google.com/github/amirgroup-codes/ProGenMech/blob/main/ProGenMech.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ProGenMech: Protein Circuit Tracing via Cross-layer Transcoders in ProGen3

ProGenMech is a framework for discovering computational circuits in the autoregressive protein language model ProGen3 using cross-layer transcoders (CLTs). This Colab notebook is designed to produce the files required for our circuit visualization website:
1. `activation_indices.json`
2. `seq.txt` (or `generation.fasta` for CLM tasks)
3. `top_activations.json`
4. `virtual_weights.json`

Unlike ProtoMech (ESM2), ProGenMech uses ProGen3 with specialized dependencies (`external/progen3`, `megablocks`, `triton`, `flash-attn`).

---

In [ ]:
# @title 0. Check GPU status and install dependencies
import torch
if torch.cuda.is_available():
    print(f"GPU Detected: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected. Code may run extremely slowly.")
    print("----------------------------------------------------------------")
    print("TO FIX THIS:")
    print("1. Click 'Runtime' in the top menu.")
    print("2. Select 'Change runtime type'.")
    print("3. Under 'Hardware accelerator', select 'T4 GPU' (or better).")
    print("4. Click 'Save' and then re-run this cell.")
    print("----------------------------------------------------------------")

import os
import sys
import pty
import shutil
import subprocess
import argparse
if hasattr(torch.serialization, 'add_safe_globals'):
    torch.serialization.add_safe_globals([argparse.Namespace])

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import ipywidgets as widgets
from IPython.display import display

# Core Python dependencies
!pip install -q pytorch-lightning pandas scipy einops transformers tqdm ipywidgets

# ProGen3-specific dependencies (megablocks + flash-attn can take several minutes)
!pip install -q "megablocks[gg]==0.7.0" grouped-gemm stanford-stk --no-build-isolation
!MAX_JOBS=4 pip install -q flash-attn==2.7.4.post1 --no-build-isolation

print("Dependency installation complete.")

In [ ]:
# @title 1. Set up ProGenMech (local repository — no GitHub/HuggingFace downloads)
# For prototyping, we assume you already have access to the full ProGenMech repository
# (including `models/`, `external/progen3/`, and `visualization/top10_activations.pt`).

TARGET_DIR = "/content/ProGenMech"  # @param {type:"string"}

# @markdown How to provide the repository:
setup_mode = "Use existing folder"  # @param ["Use existing folder", "Upload zip file"]

if setup_mode == "Upload zip file":
    if not IN_COLAB:
        raise RuntimeError("Zip upload is only supported in Google Colab.")
    print("Please upload a zip of the ProGenMech repository...")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("Upload cancelled.")
    zip_name = list(uploaded.keys())[0]
    if os.path.exists(TARGET_DIR):
        shutil.rmtree(TARGET_DIR)
    os.makedirs(TARGET_DIR, exist_ok=True)
    !unzip -q {zip_name} -d {TARGET_DIR}
    # Handle common zip layouts (repo root or nested folder)
    nested = os.path.join(TARGET_DIR, "ProGenMech")
    if os.path.isdir(nested) and os.path.isdir(os.path.join(nested, "external")):
        for item in os.listdir(nested):
            src = os.path.join(nested, item)
            dst = os.path.join(TARGET_DIR, item)
            if os.path.exists(dst):
                if os.path.isdir(dst):
                    shutil.rmtree(dst)
                else:
                    os.remove(dst)
            shutil.move(src, dst)
        shutil.rmtree(nested)
    print(f"Extracted repository to {TARGET_DIR}")
else:
    if not os.path.isdir(TARGET_DIR):
        raise FileNotFoundError(
            f"Directory {TARGET_DIR} not found. Upload a zip or set TARGET_DIR to your local ProGenMech path."
        )
    print(f"Using existing repository at {TARGET_DIR}")

# Add repo paths for imports
if TARGET_DIR not in sys.path:
    sys.path.append(TARGET_DIR)
PROGEN3_SRC = os.path.join(TARGET_DIR, "external", "progen3", "src")
if PROGEN3_SRC not in sys.path:
    sys.path.append(PROGEN3_SRC)

# Install vendored ProGen3 package
PROGEN3_ROOT = os.path.join(TARGET_DIR, "external", "progen3")
if os.path.isdir(PROGEN3_ROOT):
    !pip install -q -e {PROGEN3_ROOT}
else:
    raise FileNotFoundError(f"Missing external/progen3 at {PROGEN3_ROOT}")

# Verify required local assets (no HuggingFace downloads)
MODELS_DIR = os.path.join(TARGET_DIR, "models")
VISUALIZATION_DIR = os.path.join(TARGET_DIR, "visualization")
clt_checkpoint = os.path.join(MODELS_DIR, "ProGen3_CLT_L10_D4608", "checkpoints", "last.ckpt")
activations_pt = os.path.join(VISUALIZATION_DIR, "top10_activations.pt")

os.environ["CLT_CHECKPOINT"] = clt_checkpoint
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

if not os.path.exists(clt_checkpoint):
    raise ValueError(f"CLT checkpoint not found at {clt_checkpoint}. Ensure models/ is included in your repository copy.")
if not os.path.exists(activations_pt):
    print(f"Warning: {activations_pt} not found. Global reference activations will be unavailable.")

print(f"CLT checkpoint: {clt_checkpoint}")
print(f"Reference activations: {activations_pt}")
print("ProGenMech setup complete.")

### 1.5 Discover your own circuit
Use this section to discover a circuit and generate website-ready files.

Select a task mode:
- **CLM**: Provide a single protein prompt sequence. ProGen3 autoregressively generates the continuation and we discover the generation circuit.
- **Zero-shot**: Upload a regression-style CSV and provide sequences to score/visualize.

### **CLM input**
* **Sequence:** A single protein sequence used as the autoregressive prompt.
* **Optional:** `pos` (1-indexed) and `n_generated` to control where generation starts and how many residues are generated.

### **Zero-shot input CSV format (regression only)**
Your CSV must have two specific columns (not case-sensitive):
1. **Sequence**: `sequence` or `mutated_sequence`.
2. **Score**: `score`, `DMS_score`, or `target` (continuous numeric values).

**Example:**
```csv
mutant,mutated_sequence,DMS_score
K3R,MSR...LYK,3.74
K3Q,MSQ...LYK,3.75
K3E,MSE...LYK,3.67
```

After discovery, add the sequences you want to score (same length as each other; first sequence is treated as wildtype/reference for differential analysis).

In [ ]:
# @title 1.5 Configure circuit discovery
# @markdown **Settings**

# @markdown Select the task mode:
task_mode = "CLM"  # @param ["CLM", "Zero-shot"]

# @markdown Output folder name:
output_dir = "custom_circuit"  # @param {type:"string"}
# @markdown Where to save the results:
external_path = "/content/experiments"  # @param {type:"string"}
# @markdown Entry name for output JSON:
entry_name = "experiment"  # @param {type:"string"}

# CLM-specific settings
# @markdown CLM prompt sequence (single sequence):
clm_sequence = "MVKQVDFAEVKLSEKFLGAGSGGAVRKATFQNQEIAVKIFDFLEETIKKNAEREITHLSEIDHENVIRVIGRASNGKKDYLLMEYLEEGSLHNYLYGDDKWEYTVEQAVRWALQCAKALAYLHSLDRPIVHR"  # @param {type:"string"}
# @markdown 1-indexed generation start position (leave 0 to append at end):
clm_pos = 0  # @param {type:"integer"}
# @markdown Number of amino acids to generate:
n_generated = 5  # @param {type:"integer"}

# Zero-shot-specific settings
# @markdown Check to upload your regression CSV (Zero-shot only):
upload_csv = True  # @param {type:"boolean"}
# @markdown Optional local CSV path (for non-Colab runs; leave empty to upload):
zero_shot_csv_path = ""  # @param {type:"string"}

REPO_ROOT = TARGET_DIR
SCRIPT_PATH = os.path.join(REPO_ROOT, "visualization", "auto_discover_circuit_website.py")
EDGE_SCRIPT = os.path.join(REPO_ROOT, "visualization", "get_edge_weights.py")
full_output_dir = os.path.join(external_path, output_dir)
os.makedirs(full_output_dir, exist_ok=True)

DISCOVERY_OUTPUT_DIR = output_dir
DISCOVERY_FULL_OUTPUT_DIR = full_output_dir
DISCOVERY_ENTRY_NAME = entry_name

if task_mode == "CLM":
    if not clm_sequence.strip():
        raise ValueError("CLM mode requires a single prompt sequence.")
    print(f"CLM prompt sequence length: {len(clm_sequence.strip())}")
else:
    print("Zero-shot mode selected. Run the next cell to add sequences, then run discovery.")

In [ ]:
# @title 2. Add sequences to score (Zero-shot only; run after section 1.5)
# @markdown Example sequences (GRB2-SH3 domain):
# @markdown - Seq 1 (wildtype): `TYVQALFDFDPQEDGELGFRRGDFIHVMDNSDPNWWKGACHGQTGMFPRNYVTPVNRNV`
# @markdown - Seq 2: `TYVQALFDFDPQEDGELGFRRGDFIEVMDNSDPNWWKGACHGQTGMFPRNYVTPVNRNV`
# @markdown - Seq 3: `TYVQALFDFDPQEDGELGFRRGDFIHVMDNSDPNWWKGACHGQTGMFPRNDVTPVNRNV`
# @markdown
# @markdown Note: Use `Add Sequence +` to compare variations of the same protein. For CLM mode, skip this cell and run discovery directly.

class SequenceInputManager:
    def __init__(self):
        self.sequences = []
        self.container = widgets.VBox()
        self.add_button = widgets.Button(description="Add Sequence +", icon="plus")
        self.add_button.on_click(self.add_field)
        self.add_field(None)

    def add_field(self, b):
        idx = len(self.sequences) + 1
        label = "Seq 1 (wildtype):" if idx == 1 else f"Seq {idx}:"
        text = widgets.Text(placeholder=f"Enter protein sequence {idx}...", layout=widgets.Layout(width='80%'))
        box = widgets.HBox([widgets.Label(value=label, layout=widgets.Layout(width='120px')), text])
        self.sequences.append(text)
        self.container.children = tuple(list(self.container.children) + [box])

    def get_sequences(self):
        return [w.value.strip() for w in self.sequences if w.value.strip()]

    def display(self):
        display(widgets.VBox([self.container, self.add_button]))

seq_manager = SequenceInputManager()
seq_manager.display()

In [ ]:
# @title 3. Generate circuit files (run after configuring task and adding sequences)
# @markdown Runs `auto_discover_circuit_website.py` and then computes `virtual_weights.json`.

def run_realtime(cmd):
    master, slave = pty.openpty()
    p = subprocess.Popen(cmd, stdout=slave, stderr=slave, close_fds=True)
    os.close(slave)
    try:
        while True:
            try:
                data = os.read(master, 1024).decode()
                if not data:
                    break
                sys.stdout.write(data)
                sys.stdout.flush()
            except OSError:
                break
    except Exception:
        pass
    p.wait()
    os.close(master)
    return p.returncode


def resolve_zero_shot_csv():
    if zero_shot_csv_path.strip():
        csv_path = zero_shot_csv_path.strip()
        if not os.path.exists(csv_path):
            raise FileNotFoundError(f"CSV not found: {csv_path}")
        return csv_path
    if not upload_csv:
        raise RuntimeError("Check 'upload_csv' or set 'zero_shot_csv_path' to proceed.")
    if not IN_COLAB:
        raise RuntimeError("CSV upload requires Google Colab unless zero_shot_csv_path is set.")
    print("Please upload your regression CSV file...")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("Upload cancelled.")
    csv_name = list(uploaded.keys())[0]
    return os.path.join(os.getcwd(), csv_name)


cmd = [
    "python", SCRIPT_PATH,
    "--output_dir", full_output_dir,
    "--entry_name", entry_name,
    "--max_nodes", "1000",
    "--step_size", "32",
]

if task_mode == "CLM":
    cmd += ["--type", "CLM", "--seq", clm_sequence.strip(), "--n_generated", str(n_generated)]
    if clm_pos and clm_pos > 0:
        cmd += ["--pos", str(clm_pos)]
else:
    csv_path = resolve_zero_shot_csv()
    cmd += ["--type", "zero_shot", "--csv_path", csv_path]
    sequences = seq_manager.get_sequences()
    if sequences:
        lengths = [len(s) for s in sequences]
        if len(set(lengths)) != 1:
            raise ValueError("All sequences must have the same length.")
        cmd += ["--sequences"] + sequences

print(f"\nStarting Discovery ({task_mode})...")
print(f"   Output: {full_output_dir}/{entry_name}.json\n")
exit_code = run_realtime(cmd)

if exit_code != 0:
    print("\nDiscovery Failed.")
else:
    print("\n[Step 2] Computing virtual edge weights (may take some time)...")
    folders = []
    if os.path.isdir(full_output_dir):
        for name in sorted(os.listdir(full_output_dir)):
            path = os.path.join(full_output_dir, name)
            if os.path.isdir(path) and name.startswith("seq"):
                folders.append(path)
    has_root_inputs = (
        os.path.exists(os.path.join(full_output_dir, "generation.fasta"))
        or os.path.exists(os.path.join(full_output_dir, "seq.txt"))
    )
    if has_root_inputs:
        folders.insert(0, full_output_dir)

    for folder_path in folders:
        folder_name = os.path.basename(folder_path)
        print(f"\n   Processing {folder_name}...")
        weight_cmd = [
            "python", EDGE_SCRIPT,
            "--base_folder", folder_path,
            "--clt_ckpt", clt_checkpoint,
        ]
        w_exit = run_realtime(weight_cmd)
        if w_exit != 0:
            print(f"Failed to compute weights for {folder_name}")

    print("\n==========")
    print(f"Results saved to: {full_output_dir}")
    print("==========")
    print("Files generated:")
    for f in sorted(os.listdir(full_output_dir)):
        suffix = "/" if os.path.isdir(os.path.join(full_output_dir, f)) else ""
        print(f" - {f}{suffix}")